# 03 — Test Fine-tuned Model

Prepare data, inspect training format, and test inference with the fine-tuned model.

In [ ]:
import sys
sys.path.insert(0, "..")

import json
from pathlib import Path

## Prepare Training Data

In [ ]:
from approach2_finetune.prepare_data import prepare_chatml

prepare_chatml(
    input_path=Path("../data/samples/training_data.jsonl"),
    output_path=Path("../data/samples/prepared"),
)

In [ ]:
# Inspect a training example
with open("../data/samples/prepared/train.jsonl") as f:
    example = json.loads(f.readline())

for msg in example["messages"]:
    print(f"--- {msg['role'].upper()} ---")
    print(msg["content"][:200])
    print()

## Test Inference (requires vLLM server running)

Start the server first:
```bash
python -m approach2_finetune.serve_vllm --model outputs/merged_model
```

In [ ]:
import httpx
from shared.prompt_templates import SYSTEM_PROMPT

async def generate_with_finetune(description: str) -> str:
    payload = {
        "model": "deepseek-ai/deepseek-coder-6.7b-instruct",
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": description},
        ],
        "temperature": 0.1,
        "max_tokens": 2048,
    }
    async with httpx.AsyncClient(timeout=120.0) as client:
        resp = await client.post("http://localhost:8000/v1/chat/completions", json=payload)
        resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"]

result = await generate_with_finetune(
    "Calculate average order value per customer segment for orders in 2024"
)
print(result)

## Run Evaluation

In [ ]:
from shared.evaluation.evaluator import load_evaluation_data, evaluate_predictions, print_report

eval_examples = load_evaluation_data(Path("../data/samples/evaluation_data.jsonl"))

# Generate predictions for each eval example
predictions = []
for ex in eval_examples:
    pred = await generate_with_finetune(ex.description)
    predictions.append(pred)
    print(f"Generated for: {ex.description[:60]}...")

# Evaluate
results = evaluate_predictions(eval_examples, predictions)
print_report(results)